In [1]:
import pandas as pd
import re
from collections import Counter
import seaborn as sns

%load_ext autoreload
%autoreload 2
import src.title_versions as tv

In [2]:
baseline_df = pd.read_json('../data/baseline_2025-06-26/PMC008xxxxxx_noncomm.json')

In [3]:
baseline_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 185530 entries, 0 to 185529
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   article-type    185530 non-null  object 
 1   language        92879 non-null   object 
 2   journal         185530 non-null  object 
 3   pmc-id          185530 non-null  object 
 4   pmid            155910 non-null  float64
 5   title           185523 non-null  object 
 6   country         160471 non-null  object 
 7   date            185530 non-null  object 
 8   abstract        185530 non-null  object 
 9   section_titles  170086 non-null  object 
 10  sections        185530 non-null  object 
dtypes: float64(1), object(10)
memory usage: 17.0+ MB


In [4]:
baseline_df.head()

,article-type,language,journal,pmc-id,pmid,title,country,date,abstract,section_titles,sections
0,correction,None,Diabetes Therapy,PMC8991299,35286609.0,Correction to: Overt Diabetes in Pregnancy,India,14-3-2022,,[Correction to: Diabetes Ther 10.1007/s13300-0...,[Correction to: Diabetes Ther 10.1007/s13300-0...
1,research-article,en,International Journal of Chronic Obstructive P...,PMC8749770,35027824.0,Effect of Smoking and Its Cessation on the Tra...,India,05-1-2022,Rationale Smoking is the primary cause of chro...,"[Introduction, Methods, Results, Discussion, C...",[Introduction Chronic Obstructive Pulmonary Di...
2,research-article,en,Journal of Clinical Laboratory Analysis,PMC8605134,34528313.0,Evaluation of fungal contamination and ochrato...,"Iran, Islamic Republic of",15-9-2021,Abstract Background Mycotoxins are secondary f...,"[INTRODUCTION, MATERIALS, RESULT, DISCUSSION, ...",[1 INTRODUCTION Coffee is a member of Coffea g...
3,review-article,en,International Journal of Clinical Pediatric De...,PMC8783216,NaN,The Epiphany of Post-COVID: A Watershed for Pe...,India,Nov-Dec-2021,A bstract Coronavirus disease-2019 (COVID-19) ...,"[I, E, T, S, D, C]",[I ntroduction Coronavirus disease-2019 (COVID...
4,research-article,None,Schizophrenia Research: Cognition,PMC8131976,34026572.0,Screening for cognitive impairment in schizoph...,Austria,12-5-2021,Background The Screen for Cognitive Impairment...,"[Introduction, Methods and materials, Results,...",[1 Introduction Cognitive impairment is consid...


In [5]:
#replace None with empty string to make my life easier
baseline_df['section_titles'] = baseline_df['section_titles'].apply(
    lambda x: [] if x == None else x
)

In [16]:
baseline_df[baseline_df['section_titles'].apply(lambda x: not x == None  and 'Expected outcomes' in x)]

,article-type,language,journal,pmc-id,pmid,title,country,date,abstract,section_titles,sections,section_titles_cleaned
27,research-article,None,STAR Protocols,PMC8384916,34467229.0,Induction of granule and Purkinje cells from p...,France,18-8-2021,Summary The architecturally stereotypical stru...,"[Before you begin, Key resources table, Materi...",[Before you begin The following protocol descr...,[limitations]
70,research-article,None,STAR Protocols,PMC8369072,34430916.0,Electrophysiological and anatomical characteri...,Japan,13-8-2021,"Summary In the central nervous system, develop...","[Before you begin, Key resources table, Materi...",[Before you begin Structures and functions of ...,[limitations]
232,research-article,None,STAR Protocols,PMC8273411,34286292.0,Building predictive signaling models by pertur...,United States of America,07-7-2021,Summary This protocol provides a step-by-step ...,"[Before you begin, Key resources table, Materi...",[Before you begin Timing: 1 day Cells in their...,[limitations]
243,research-article,None,STAR Protocols,PMC8689352,34977685.0,Purification and preparation of Rhodobacter s...,Canada,16-12-2021,Summary The formation of defined surfaces cons...,"[Before you begin, Key resources table, Materi...","[Before you begin In this protocol, we begin w...",[limitations]
567,research-article,None,STAR Protocols,PMC8044720,33870228.0,Protocol for building a cognitive map of struc...,United States of America,30-3-2021,Summary Humans are adept at learning the laten...,"[Before you begin, Key resources table, Step-b...",[Before you begin Stimuli 1. To create a socia...,[limitations]
...,...,...,...,...,...,...,...,...,...,...,...,...
184086,research-article,None,STAR Protocols,PMC8672044,34950882.0,Induction of input-specific spine shrinkage on...,United States of America,11-12-2021,Summary Shrinkage and loss of dendritic spines...,"[Before you begin, Key resources table, Materi...",[Before you begin The protocol below describes...,[limitations]
184199,research-article,None,STAR Protocols,PMC8243151,34223199.0,"Ultracentrifugal separation, characterization,...",Japan,23-6-2021,Summary Extracellular vesicles (EVs) play impo...,"[Before you begin, Key resources table, Materi...",[Before you begin EVs are vesicles naturally s...,[limitations]
184248,research-article,None,STAR Protocols,PMC8882418,NaN,SARS-CoV-2 proteome microarray for COVID-19 pa...,China,28-2-2022,Summary The immunogenicity of severe acute res...,"[Before you begin, Key resources table, Materi...",[Before you begin This protocol was used in a ...,[limitations]
184725,research-article,None,STAR Protocols,PMC8379521,34458864.0,Annotating cell types in human single-cell RNA...,United States of America,17-8-2021,Summary Cell type annotation is important in t...,"[Before you begin, Key resources table, Step-b...",[Before you begin Cell type annotation is an i...,[limitations]


In [ ]:
def get_alt_title(title: str, title_versions: dict) -> str:
    
    # sections we're not interested in
    ignore_list = ['supplementary', 'admin', 'combined', 'misc'] 
    
    # if title is not in the inclusion list, set to empty string
    alt = ''
    for key, synonyms in title_versions.items():
        if key in ignore_list:
            continue
        if not title == None:
            # clean title
            for pat, repl in tv.title_clean.items():
                title = re.sub(pat, repl, title)
            title = title.strip(':').strip()
            if title.casefold() in synonyms:
                alt = key
    return alt

In [8]:
def replace_section_titles(title_versions: dict, section_titles):

    alt_titles = [[get_alt_title(title, title_versions) for title in secs] 
                    for secs in section_titles]
    alt_titles = [[title for title in sorted(secs) if not title == '']
                  for secs in alt_titles]

    return alt_titles
    


In [9]:
baseline_df['section_titles_cleaned'] = replace_section_titles(tv.title_versions, list(baseline_df['section_titles']))

In [10]:
for i in range(10, 20):
    print(f'before: {baseline_df['section_titles'].iloc[i]}')
    print(f'after: {baseline_df['section_titles_cleaned'].iloc[i]}')

before: ['Le polysulfate de pentosan est le principal traitement de la douleur vésicale associée à la cystite interstitielle', 'La maculopathie est associée à une utilisation prolongée du polysulfate de pentosan', 'La maculopathie causée par le polysulfate de pentosan peut s’apparenter à la dégénérescence maculaire liée à l’âge', 'La maladie maculaire peut poursuivre sa progression malgré l’arrêt du polysulfate de pentosan', 'Les patients exposés au polysulfate de pentosan qui rapportent des troubles de la vision devraient subir un dépistage ophtalmique']
after: []
before: ['INTRODUCTION', 'BACKGROUND', 'THE STUDY', 'RESULTS', 'DISCUSSION', 'CONCLUSIONS', 'CONFLICT OF INTEREST', 'AUTHOR CONTRIBUTIONS', 'ETHICAL APPROVAL AND PATIENT CONSENT\u202fFOR PUBLICATION\u202fSTATEMENT']
after: ['background', 'conclusion', 'discussion', 'introduction', 'results']
before: []
after: []
before: ['Introduction', 'Case Report', 'Discussion']
after: ['case report', 'discussion', 'introduction']
before:

In [11]:
Counter(map(str, baseline_df['section_titles_cleaned']))

Counter({"['conclusion', 'discussion', 'introduction', 'methods', 'results']": 38846,
         "['discussion', 'introduction', 'methods', 'results']": 37718,
         '[]': 29835,
         "['conclusion', 'introduction']": 11543,
         "['case report', 'conclusion', 'discussion', 'introduction']": 6265,
         "['discussion', 'methods', 'results']": 6056,
         "['case report', 'discussion', 'introduction']": 5530,
         "['conclusion', 'introduction', 'methods']": 5089,
         "['conclusion', 'discussion', 'methods', 'results']": 4634,
         "['conclusion', 'experiment', 'introduction']": 3445,
         "['introduction']": 3129,
         "['conclusion']": 2094,
         "['background', 'conclusion', 'discussion', 'methods', 'results']": 1992,
         "['conclusion', 'discussion', 'introduction', 'limitations', 'methods', 'results']": 1953,
         "['discussion', 'introduction']": 1546,
         "['introduction', 'methods']": 1532,
         "['conclusion', 'discussio

In [12]:
# proportion of all papers where no titles were found
len(baseline_df[baseline_df['section_titles'].apply(lambda x: x == [])]) / len(baseline_df)

0.08324260227456476

In [13]:
# proportion of research articles in batch
article_count = len(baseline_df[baseline_df['article-type'] == 'research-article']) 
article_count / len(baseline_df)

0.6443486228642268

In [14]:
# proportion of research articles where no titles were found
len(baseline_df[baseline_df['section_titles'].apply(lambda x: x == [])][baseline_df['article-type'] == 'research-article']) / article_count

/tmp/ipykernel_1114804/1322135802.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  len(baseline_df[baseline_df['section_titles'].apply(lambda x: x == [])][baseline_df['article-type'] == 'research-article']) / article_count


0.00964482291335553

In [15]:
Counter(baseline_df['article-type'])

Counter({'research-article': 119546,
         'case-report': 18783,
         'review-article': 16340,
         'letter': 5391,
         'abstract': 5157,
         'editorial': 4767,
         'brief-report': 4645,
         'other': 4090,
         'correction': 1575,
         'discussion': 1281,
         'article-commentary': 1218,
         'systematic-review': 708,
         'data-paper': 438,
         'news': 395,
         'retraction': 315,
         'reply': 213,
         'rapid-communication': 178,
         'meeting-report': 131,
         'report': 94,
         'book-review': 61,
         'methods-article': 53,
         'obituary': 45,
         'introduction': 39,
         'expression-of-concern': 26,
         'in-brief': 15,
         'product-review': 11,
         'addendum': 9,
         'announcement': 2,
         'oration': 2,
         ' research-article': 1,
         'protocol': 1})